# Tornado Diagram Plot

    JMA 3 Aug 2026

In [1]:
import os, sys
import numpy as np
import pandas as pd

Create a plot of a tornado diagram as used in decision analysis.

A tornado diagram is a horizontal bar plot showing the sensitivity of the utility function to each input variable, separately. Each variable is shown by a bar.  The  bar plots the utility function for the 10th percentile and 90th percentile of the variable, with all other variables set to their median value -- their 50th percentile.  The bars are centered at their 50th percentile utility. 

The input to the diagram is a pandas dataframe, vars,  with variables naming the rows. Probability variables have their P10, P50 and P90 values in columns.  The utility function is a python function named utility() whose arguments include all variables, and which returns a numeric value. 

The output is the table of utility ranges for the P10 and P90 values of each variable, sorted by largest range to smallest range.  This output is plotted as a horizontal bar chart. 

In [2]:
def utility(price, quantity, discount, years, initial, final, fixed):
    annual = price*quantity - fixed
    yrs = np.ones(years) * annual
    yrs[0] = yrs[0] - initial
    yrs[years-1] = yrs[years-1] + final
    rate = np.ones(years) * (1-discount)
    rates = np.cumprod(rate)
    cash_flow = np.multiply(yrs, rates)
    return cash_flow.sum()


In [3]:
# Test utility function
[utility(100, 99, 0.2, 5, 20000,k,  5000) for k in np.linspace(0, 20000, 10)]

[np.float64(-2822.5279999999975),
 np.float64(-2094.350222222219),
 np.float64(-1366.1724444444408),
 np.float64(-637.9946666666633),
 np.float64(90.18311111111507),
 np.float64(818.360888888893),
 np.float64(1546.538666666671),
 np.float64(2274.716444444449),
 np.float64(3002.8942222222267),
 np.float64(3731.0720000000056)]

In [4]:
intervals = [
    {'var':'price', 'P10':10 ,  'P50':100 , 'P90':200 },
    {'var':'quantity', 'P10':99 ,  'P50':199 , 'P90':299 },
    {'var':'discount' ,'P10':0.0 ,  'P50':0.03 , 'P90':0.12 },
    {'var':'years' ,'P10':4 ,  'P50':10 , 'P90':20 },
    {'var':'initial' ,'P10':5000 ,  'P50':10000 , 'P90':25000 },
    {'var':'final' ,'P10':10000 ,  'P50':20000 , 'P90':60000 },
    {'var':'fixed' ,'P10':3000 ,  'P50':5000 , 'P90':8000 },
]

vars = pd.DataFrame(intervals).set_index('var')
vars


,P10,P50,P90
var,,,
price,10.0,100.00,200.00
quantity,99.0,199.00,299.00
discount,0.0,0.03,0.12
years,4.0,10.00,20.00
initial,5000.0,10000.00,25000.00
final,10000.0,20000.00,60000.00
fixed,3000.0,5000.00,8000.00


## Build the utility-range table

For each row in `vars`, evaluate `utility` at three points while holding every other variable at its P50 (median):

* `U_P10`  = utility at the row's P10, all others at P50
* `U_base`= utility at the row's P50, all others at P50  (same for every row)
* `U_P90`  = utility at the row's P90, all others at P50

The bar for the variable spans from `U_P10` to `U_P90` and is centered on `U_base`. The table is sorted by absolute range (|U_P90 - U_P10|) so the most influential variables appear at the top of the chart.

In [5]:
INT_VARS = {'years'}  # variables that must be passed as int to utility()

def build_utility_table(vars_df, utility):
    """Return a DataFrame of utility swings per variable.

    For each row in vars_df (index = variable name; columns P10,
    P50, P90), evaluate utility at three points with every other
    variable held at P50. Sort by absolute range, largest first.
    """
    def _coerce(values):
        out = {}
        for k, v in values.items():
            out[k] = int(v) if k in INT_VARS else float(v)
        return out

    var_names = list(vars_df.index)
    base = _coerce({name: vars_df.loc[name, 'P50'] for name in var_names})
    u_base = float(utility(**base))

    rows = []
    for name in var_names:
        p10 = dict(base); p10[name] = _coerce({name: vars_df.loc[name, 'P10']})[name]
        p90 = dict(base); p90[name] = _coerce({name: vars_df.loc[name, 'P90']})[name]
        u_p10 = float(utility(**p10))
        u_p90 = float(utility(**p90))
        swing = u_p90 - u_p10
        rows.append({
            'Variable': name,
            'P10': float(vars_df.loc[name, 'P10']),
            'P50': float(vars_df.loc[name, 'P50']),
            'P90': float(vars_df.loc[name, 'P90']),
            'U_P10': u_p10,
            'U_base': u_base,
            'U_P90': u_p90,
            'Range': swing,
            'Range_abs': abs(swing),
        })
    table = pd.DataFrame(rows)
    return table.sort_values('Range_abs', ascending=False).reset_index(drop=True)

table = build_utility_table(vars, utility)
table

,Variable,P10,P50,P90,U_P10,U_base,U_P90,Range,Range_abs
0,price,10.0,100.00,200.00,-20506.276686,131548.785671,300498.854956,321005.131641,321005.131641
1,quantity,99.0,199.00,299.00,46649.253367,131548.785671,216448.317975,169799.064608,169799.064608
2,years,4.0,10.00,20.00,63267.956769,131548.785671,220960.565581,157692.608812,157692.608812
3,discount,0.0,0.03,0.12,159000.000000,131548.785671,75605.812875,-83394.187125,83394.187125
4,fixed,3000.0,5000.00,8000.00,148528.692132,131548.785671,106078.925980,-42449.766152,42449.766152
5,final,10000.0,20000.00,60000.00,124174.544402,131548.785671,161045.750747,36871.206345,36871.206345
6,initial,5000.0,10000.00,25000.00,136398.785671,131548.785671,116998.785671,-19400.000000,19400.000000


In [6]:
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, Label, Span
from bokeh.plotting import figure

output_notebook()

# Largest swing on top, so reverse the y-range.
y_vars = list(table['Variable'])[::-1]
u_p10 = [float(v) for v in table['U_P10']][::-1]
u_p90 = [float(v) for v in table['U_P90']][::-1]
u_base = float(table['U_base'].iloc[0])
labels_p10 = [f'{v:.3g}' for v in u_p10]
labels_p90 = [f'{v:.3g}' for v in u_p90]

p = figure(
    title='Tornado diagram: NPV sensitivity to each input variable',
    y_range=y_vars,
    width=750, height=420,
    x_axis_label='Utility (NPV)',
    y_axis_label='Variable',
    toolbar_location=None,
    background_fill_color='#fafafa',
)

# Bars: left = min(P10, P90), right = max(P10, P90). Negative-width
# bars are not needed here because we take the bracket explicitly.
left = [min(a, b) for a, b in zip(u_p10, u_p90)]
right = [max(a, b) for a, b in zip(u_p10, u_p90)]
p.hbar(
    y=y_vars, left=left, right=right,
    height=0.55,
    color='#6baed6', alpha=0.8,
    line_color='#08519c', line_width=1,
)

# Vertical reference line at the base (all-P50) utility.
base_span = Span(
    location=u_base, dimension='height',
    line_color='firebrick', line_width=2, line_dash='dashed',
)
p.add_layout(base_span)
p.add_layout(Label(
    x=u_base, y=len(y_vars) - 0.4,
    text=f'Base utility = {u_base:.3g}',
    text_color='firebrick', text_font_size='9pt', x_offset=4,
))

# End-of-bar value labels (P10 on the left, P90 on the right).
p.text(x=u_p10, y=y_vars, text=labels_p10, x_offset=-4,
       text_align='right', text_baseline='middle',
       text_font_size='9pt', text_color='#08519c')
p.text(x=u_p90, y=y_vars, text=labels_p90, x_offset=4,
       text_align='left', text_baseline='middle',
       text_font_size='9pt', text_color='#08519c')

p.grid.grid_line_alpha = 0.3
show(p)

Loading BokehJS ...